In [2]:
# ============================================================
# Cell 1 — Odometry Quality Assessment (fixed)
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

FILES = {
    1: "square_1m_r1.csv",
    2: "square_1m_r2.csv",
    3: "square_1m_r3.csv",
    4: "square_1m_r4.csv",
}

# ---------- Load ----------
print("=" * 92)
print(" ODOMETRY QUALITY — 1×1 m Square Maneuver")
print("=" * 92)

data = {}
for r, fname in FILES.items():
    p = Path(fname)
    if not p.exists():
        print(f"  MISSING: {fname}")
        continue
    d = pd.read_csv(p)
    data[r] = d
    print(f"  R{r}: {len(d):>5d} rows, "
          f"duration {(d['elapsed_ms'].iloc[-1]/1000):>5.1f} s, "
          f"states {sorted(d['state'].unique())}")

# ---------- 1. LOOP CLOSURE ----------
print("\n" + "=" * 92)
print(" [1] LOOP CLOSURE ERROR  (robot must return to start pose)")
print("=" * 92)
print(f"\n{'robot':>5s} | {'X_end [m]':>10s} | {'Y_end [m]':>10s} | "
      f"{'|err| [cm]':>10s} | {'Yaw_end [°]':>11s} | {'Yaw_err [°]':>11s}")
print("-" * 74)

loop_results = []
for r, d in data.items():
    last = d.iloc[-1]
    xe, ye = last["pos_x_m"], last["pos_y_m"]
    err = np.hypot(xe, ye) * 100
    yaw_end = last["yaw_deg"]
    yaw_err = ((yaw_end + 180) % 360) - 180
    print(f"{r:>5d} | {xe:>+10.4f} | {ye:>+10.4f} | "
          f"{err:>10.2f} | {yaw_end:>+11.2f} | {yaw_err:>+11.2f}")
    loop_results.append({
        "robot": r, "x_end_m": xe, "y_end_m": ye,
        "loop_error_cm": err, "yaw_end_deg": yaw_end,
        "yaw_error_deg": yaw_err,
    })

# ---------- 2. DRIFT RATE ----------
print("\n" + "=" * 92)
print(" [2] DRIFT RATE  (loop_error / total_path)")
print("=" * 92)

R_WHEEL = 0.02685
CPR     = 4096.0
TICK_M  = (2 * np.pi * R_WHEEL) / CPR

print(f"\n{'robot':>5s} | {'path [m]':>9s} | {'|err| [cm]':>10s} | "
      f"{'drift [cm/m]':>13s} | {'verdict':>11s}")
print("-" * 66)

for res in loop_results:
    r = res["robot"]
    d = data[r]
    steps_l = d["steps_l"].iloc[-1]
    steps_r = d["steps_r"].iloc[-1]
    path_l = steps_l * TICK_M
    path_r = steps_r * TICK_M
    path_avg = 0.5 * (path_l + path_r)
    drift = res["loop_error_cm"] / path_avg if path_avg > 0 else np.nan
    if drift < 2:
        v = "EXCELLENT"
    elif drift < 5:
        v = "GOOD"
    elif drift < 10:
        v = "MARGINAL"
    else:
        v = "POOR"
    print(f"{r:>5d} | {path_avg:>9.3f} | {res['loop_error_cm']:>10.2f} | "
          f"{drift:>13.2f} | {v:>11s}")
    res["path_m"] = path_avg
    res["drift_cm_per_m"] = drift

# ---------- 3. TOTAL ROTATION (unwrapped) ----------
print("\n" + "=" * 92)
print(" [3] TOTAL ROTATION  (unwrapped yaw over the whole maneuver)")
print("=" * 92)
print(" Expected: four CCW 90° turns → +360°. Measured via unwrap of yaw.")
print(f"\n{'robot':>5s} | {'yaw_first [°]':>13s} | {'yaw_last [°]':>13s} | "
      f"{'total rot [°]':>14s} | {'error [°]':>10s}")
print("-" * 68)

def unwrap_deg(yaw_deg):
    """Unwrap a yaw series expressed in degrees; assumes monotone motion."""
    y = np.asarray(yaw_deg, dtype=float)
    return np.degrees(np.unwrap(np.radians(y)))

for r, d in data.items():
    yaw_u = unwrap_deg(d["yaw_deg"].values)
    y0 = yaw_u[0]
    y1 = yaw_u[-1]
    total = y1 - y0
    err = total - 360.0
    print(f"{r:>5d} | {y0:>+13.2f} | {y1:>+13.2f} | "
          f"{total:>+14.2f} | {err:>+10.2f}")

# ---------- 4. PER-TURN ROTATION (from yaw deltas between turn states) ----------
print("\n" + "=" * 92)
print(" [4] PER-TURN ROTATION  (yaw at start of each turn state)")
print("=" * 92)
print(f"\n{'robot':>5s} | {'turn1':>8s} | {'turn2':>8s} | "
      f"{'turn3':>8s} | {'turn4':>8s} | {'sum':>8s} | {'std':>6s}")
print("-" * 66)

for r, d in data.items():
    # Use unwrapped yaw for correct delta
    yaw_u = unwrap_deg(d["yaw_deg"].values)
    d = d.copy()
    d["yaw_u"] = yaw_u
    
    turn_deltas = []
    for ts in [11, 12, 13, 14]:
        seg = d[d["state"] == ts]
        if len(seg) < 5:
            turn_deltas.append(np.nan); continue
        # Take the first and last sample of this turn state
        y0 = seg["yaw_u"].iloc[0]
        y1 = seg["yaw_u"].iloc[-1]
        turn_deltas.append(y1 - y0)
    total = np.nansum(turn_deltas)
    spread = np.nanstd(turn_deltas)
    print(f"{r:>5d} | {turn_deltas[0]:>8.2f} | {turn_deltas[1]:>8.2f} | "
          f"{turn_deltas[2]:>8.2f} | {turn_deltas[3]:>8.2f} | "
          f"{total:>8.2f} | {spread:>6.2f}")

# ---------- 5. STRAIGHT LEG LENGTH ----------
print("\n" + "=" * 92)
print(" [5] STRAIGHT LEG LENGTH  (should be exactly 1.000 m each)")
print("=" * 92)
print(f"\n{'robot':>5s} | {'leg1':>8s} | {'leg2':>8s} | "
      f"{'leg3':>8s} | {'leg4':>8s} | {'mean':>8s} | {'std [cm]':>9s}")
print("-" * 68)

for r, d in data.items():
    leg_lengths = []
    for ls in [1, 2, 3, 4]:
        seg = d[d["state"] == ls]
        if len(seg) < 5:
            leg_lengths.append(np.nan); continue
        sl0 = seg["steps_l"].iloc[0]; sl1 = seg["steps_l"].iloc[-1]
        sr0 = seg["steps_r"].iloc[0]; sr1 = seg["steps_r"].iloc[-1]
        dl = (sl1 - sl0) * TICK_M
        dr = (sr1 - sr0) * TICK_M
        leg_lengths.append(0.5 * (dl + dr))
    mean_leg = np.nanmean(leg_lengths)
    std_leg = np.nanstd(leg_lengths) * 100
    print(f"{r:>5d} | {leg_lengths[0]:>8.4f} | {leg_lengths[1]:>8.4f} | "
          f"{leg_lengths[2]:>8.4f} | {leg_lengths[3]:>8.4f} | "
          f"{mean_leg:>8.4f} | {std_leg:>9.2f}")

# ---------- 6. SLIP EVENTS ----------
print("\n" + "=" * 92)
print(" [6] SLIP EVENTS  (gating rejections in HeadingEKF)")
print("=" * 92)
print(f"\n{'robot':>5s} | {'slip samples':>13s} | {'total':>8s} | {'slip %':>8s}")
print("-" * 46)
for r, d in data.items():
    n_slip = int(d["slip"].sum())
    total = len(d)
    print(f"{r:>5d} | {n_slip:>13d} | {total:>8d} | "
          f"{100.0 * n_slip / total:>7.2f}%")

# ---------- 7. GYRO BIAS STABILITY DURING ZUPT ----------
print("\n" + "=" * 92)
print(" [7] GYRO BIAS STABILITY DURING STATIC PHASE  (state == 0)")
print("=" * 92)
print(f"\n{'robot':>5s} | {'dur [s]':>8s} | {'bias_start':>11s} | "
      f"{'bias_end':>10s} | {'Δbias':>9s} | {'yaw drift [°]':>14s}")
print("-" * 74)
for r, d in data.items():
    z = d[d["state"] == 0]
    if len(z) < 10:
        continue
    dur = (z["elapsed_ms"].iloc[-1] - z["elapsed_ms"].iloc[0]) / 1000.0
    b0 = z["bias_dps"].iloc[0]
    b1 = z["bias_dps"].iloc[-1]
    ydrift = z["yaw_deg"].iloc[-1] - z["yaw_deg"].iloc[0]
    print(f"{r:>5d} | {dur:>8.2f} | {b0:>+11.3f} | "
          f"{b1:>+10.3f} | {b1-b0:>+9.4f} | {ydrift:>+14.3f}")

# ---------- 8. VERDICT ----------
print("\n" + "=" * 92)
print(" VERDICT")
print("=" * 92)

mean_loop   = np.mean([r["loop_error_cm"] for r in loop_results])
mean_drift  = np.mean([r["drift_cm_per_m"] for r in loop_results])
max_drift   = np.max ([r["drift_cm_per_m"] for r in loop_results])
mean_yawerr = np.mean([abs(r["yaw_error_deg"]) for r in loop_results])

print(f"\n  Mean loop closure error : {mean_loop:.2f} cm")
print(f"  Mean yaw final error    : {mean_yawerr:.2f} deg")
print(f"  Mean drift rate         : {mean_drift:.2f} cm/m")
print(f"  Worst drift rate        : {max_drift:.2f} cm/m")

print()
if mean_drift < 2:
    print("  ✓ Odometry is EXCELLENT.")
    print("    → It can reliably help solve the R1 bias problem.")
elif mean_drift < 5:
    print("  ✓ Odometry is GOOD.")
    print("    → R1 bias can be estimated with odometry assistance.")
elif mean_drift < 10:
    print("  ~ Odometry is MARGINAL.")
    print("    → R1 bias will improve but not resolve fully.")
else:
    print("  ✗ Odometry is POOR.")
    print("    → Cannot distinguish R1 bias from odom drift.")

# ---------- Save ----------
pd.DataFrame(loop_results).to_csv("odometry_quality_loop.csv", index=False)
print(f"\n  Saved: odometry_quality_loop.csv")

 ODOMETRY QUALITY — 1×1 m Square Maneuver
  R1:  3419 rows, duration  34.2 s, states [0, 1, 2, 3, 4, 11, 12, 13, 14]
  R2:  3498 rows, duration  35.0 s, states [0, 1, 2, 3, 4, 11, 12, 13, 14]
  R3:  3371 rows, duration  33.7 s, states [0, 1, 2, 3, 4, 11, 12, 13, 14]
  R4:  3242 rows, duration  32.4 s, states [0, 1, 2, 3, 4, 11, 12, 13, 14]

 [1] LOOP CLOSURE ERROR  (robot must return to start pose)

robot |  X_end [m] |  Y_end [m] | |err| [cm] | Yaw_end [°] | Yaw_err [°]
--------------------------------------------------------------------------
    1 |    +0.0327 |    -0.0518 |       6.13 |     +358.94 |       -1.06
    2 |    -0.0026 |    -0.0256 |       2.57 |     +358.64 |       -1.36
    3 |    +0.0254 |    -0.0231 |       3.43 |     +358.49 |       -1.51
    4 |    +0.0362 |    +0.0020 |       3.63 |     +358.40 |       -1.60

 [2] DRIFT RATE  (loop_error / total_path)

robot |  path [m] | |err| [cm] |  drift [cm/m] |     verdict
---------------------------------------------------